# 第10章　可转换债券

[![在 Colab 打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/albertandking/fixed-income/blob/main/notebooks/ch10_convertible.ipynb) [![在 Binder 打开](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/albertandking/fixed-income/main?labpath=notebooks/ch10_convertible.ipynb)

复现例11.1（三个价值与溢价率）、例11.2（二叉树与无套利下限）、图10-1（股债性切换）、强赎影响，QuantLib 对拍与信用利差敏感性。


In [ ]:
# 自举单元：在 Colab/Binder 上自动安装本书复用包 fi；本地运行时自动跳过。
import importlib.util, sys, subprocess
if importlib.util.find_spec('fi') is None:
    if 'google.colab' in sys.modules:
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/albertandking/fixed-income.git', '/content/fi-book'], check=False)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '/content/fi-book'], check=False)
    else:
        print('提示：请在仓库根目录执行 `uv sync --extra all` 后再运行本 notebook。')


In [ ]:
import numpy as np
from fi import convertible as cb
from fi import plotting
plotting.use_chinese_style()


## 例11.1　转股价值、纯债价值与溢价率

5 年期、年付息 1.5%、面值 100、转股比例 10；σ=30%、r=3%；纯债底按 5% 信用折现。


In [ ]:
floor = cb.bond_floor(100, 0.015, 5, discount_rate=0.05)
print(f'纯债价值（bond floor）= {floor:.2f}')
print(f"{'股价':>4}{'可转债':>9}{'转股价值':>9}{'转股溢价率':>11}{'纯债溢价率':>11}")
for S in (5, 8, 12):
    res = cb.price_convertible(S, 0.30, 0.03, 5, 100, 0.015, 10, n_steps=200)
    px, parity = res['price'], res['conversion_value']
    print(f'{S:>4}{px:>9.2f}{parity:>9.1f}{(px-parity)/parity*100:>10.1f}%{(px-floor)/floor*100:>10.1f}%')


## 例11.2　二叉树定价始终高于无套利下限


In [ ]:
for S in (3, 8, 12, 15):
    px = cb.price_convertible(S, 0.30, 0.03, 5, 100, 0.015, 10, n_steps=200)['price']
    lb = max(floor, 10*S)
    print(f'S={S:2d}: 可转债={px:7.2f}  >= max(底{floor:.1f}, 转股{10*S}) = {lb:.1f}')


## 图10-1　股债性切换（编程实验 6）


In [ ]:
s = np.linspace(2, 16, 36)
cv = [cb.price_convertible(si, 0.30, 0.03, 5, 100, 0.015, 10, n_steps=120)['price'] for si in s]
fig, ax = plotting.new_axes()
ax.plot(s, cv, lw=2, label='可转债价值')
ax.plot(s, 10*s, '--', label='转股价值（ratio×S）')
ax.axhline(floor, ls=':', color='gray', label=f'纯债底 {floor:.1f}')
ax.set_xlabel('正股价格 S'); ax.set_ylabel('价值')
ax.set_title('图10-1　可转债的股债性切换'); ax.legend()
fig.tight_layout()


### 强赎条款封顶股性收益（编程实验 7）


In [ ]:
print('高股价区：有无强赎(call=105)对可转债价值的影响')
for S in (10, 12, 14, 16):
    no_call = cb.price_convertible(S, 0.30, 0.03, 5, 100, 0.015, 10, n_steps=200)['price']
    with_call = cb.price_convertible(S, 0.30, 0.03, 5, 100, 0.015, 10, n_steps=200, call_price=105)['price']
    print(f'  S={S:2d}: 无强赎={no_call:7.2f}  有强赎={with_call:7.2f}  差={no_call-with_call:5.2f}')


## 11.6　QuantLib 对拍与信用利差敏感性（编程实验 8）


In [ ]:
import QuantLib as ql
today = ql.Date(15, 6, 2026); ql.Settings.instance().evaluationDate = today
dc, cal = ql.Actual365Fixed(), ql.NullCalendar()
spot = ql.QuoteHandle(ql.SimpleQuote(8.0))
rTS = ql.YieldTermStructureHandle(ql.FlatForward(today, 0.03, dc))
qTS = ql.YieldTermStructureHandle(ql.FlatForward(today, 0.0, dc))
volTS = ql.BlackVolTermStructureHandle(ql.BlackConstantVol(today, cal, 0.30, dc))
process = ql.BlackScholesMertonProcess(spot, qTS, rTS, volTS)
exercise = ql.AmericanExercise(today, today + ql.Period(5, ql.Years))
sched = ql.Schedule(today, today + ql.Period(5, ql.Years), ql.Period(ql.Annual), cal,
                    ql.Unadjusted, ql.Unadjusted, ql.DateGeneration.Backward, False)
bond = ql.ConvertibleFixedCouponBond(exercise, 10.0, ql.CallabilitySchedule(),
                                     today, 0, [0.015], dc, sched, 100.0)
print(f'fi.convertible (无风险折现) = {cb.price_convertible(8,0.30,0.03,5,100,0.015,10,n_steps=200)["price"]:.3f}')
for cs in (0.0, 0.02, 0.04):
    bond.setPricingEngine(ql.BinomialConvertibleEngine(
        process, 'crr', 200, ql.QuoteHandle(ql.SimpleQuote(cs)), ql.DividendSchedule()))
    print(f'QuantLib 信用利差={cs*100:.0f}%: NPV = {bond.NPV():.3f}')


---

> 小结：可转债 = 纯债 + 转股权；价值始终 ≥ max(纯债底, 转股价值)，低股价像债、高股价像股；
> 信用利差越高纯债底越低、可转债越便宜——`fi.convertible` 与 QuantLib 定性一致。
